# Process Window 분석 대시보드 만들기 — Vue3 + FastAPI 입문

> 웹 개발 경험이 거의 없는 데이터 분석가를 위한 **개념 정리 + 실습 가이드** 노트북입니다.
> 이 노트북 한 권으로 "왜 이렇게 구성하는가"를 이해하고, 옆 폴더의 `backend/`, `frontend/` 코드를 실행할 수 있게 됩니다.

## 우리가 만들 것

1. **Binned Combination Chart**
   - X = 선택한 feature 값을 10개 구간으로 binning
   - 막대(bar) = 각 구간의 wafer count
   - 선(line) = 각 구간의 target(y) 평균
   - user spec `lower ~ upper`를 range filter로 정하면 → 차트에 **수직선(vertical line)** 으로 표시
2. **시계열 Dual Scatter Chart**
   - X = `fab_step`의 `trackout_time`
   - 위 차트 Y = target(y) 값
   - 아래 차트 Y = feature 값
   - 두 차트가 **시간축으로 연동**되어 같이 움직임

## 학습 로드맵 (이 노트북의 목차)

| 장 | 주제 | 핵심 질문 |
|----|------|-----------|
| 1 | 큰 그림 | 프론트엔드 / 백엔드는 어떻게 대화하는가? |
| 2 | 필수 웹 개념 | HTTP, REST API, JSON, CORS, 비동기 |
| 3 | FastAPI 기초 | 파이썬으로 API 서버 만들기 |
| 4 | 핵심 분석 로직 | pandas로 binning 계산하기 (여기서 실제 실행) |
| 5 | Vue3 기초 | 반응형(reactivity), 컴포넌트, `<script setup>` |
| 6 | 프론트-백 연계 | axios, CORS, Vite proxy |
| 7 | 차트 (ECharts) | Combination chart + spec 수직선 |
| 8 | 시계열 차트 | 두 scatter를 시간축으로 연동 |
| 9 | 파일 아키텍처 | 프로젝트 폴더 구조 |
| 10 | 실행 방법 | 실제로 띄워 보기 |

## 1. 큰 그림 — 웹 대시보드는 어떻게 동작하는가

웹 대시보드는 보통 **두 개의 프로그램**으로 나뉩니다.

```
┌─────────────────────────┐         HTTP 요청 (JSON)        ┌──────────────────────────┐
│   FRONTEND  (Vue3)       │  ───────────────────────────▶  │   BACKEND  (FastAPI)     │
│   - 브라우저에서 실행     │                                 │   - 파이썬으로 실행       │
│   - 화면(UI), 차트 그리기 │  ◀───────────────────────────  │   - 데이터 계산(binning) │
│   - 사용자 입력 받기      │         HTTP 응답 (JSON)        │   - DB / 파일 읽기        │
└─────────────────────────┘                                 └──────────────────────────┘
        :5173 (개발 서버)                                            :8000 (API 서버)
```

핵심 멘탈 모델:

- **백엔드(FastAPI)** = "데이터를 계산해서 JSON으로 돌려주는 함수 모음". 화면은 모릅니다.
- **프론트엔드(Vue3)** = "JSON을 받아서 화면/차트로 그리는 프로그램". 계산 로직은 모릅니다.
- 둘은 **HTTP + JSON** 이라는 약속(계약, contract)으로만 대화합니다.

> 💡 분석가에게 익숙하게 비유하면: 백엔드는 잘 만든 `함수(df) -> dict`이고,
> 프론트엔드는 그 dict를 받아 `matplotlib` 대신 브라우저에 그리는 부분입니다.
> 차이는 "함수 호출"이 같은 메모리가 아니라 **네트워크 너머**에서 일어난다는 점뿐입니다.

### 왜 굳이 둘로 나누나?
- 무거운 계산(수백만 wafer binning)은 서버에서, 가벼운 인터랙션은 브라우저에서 → 역할 분리
- 데이터(원본)는 서버에만 두고, 브라우저엔 "결과"만 보냄 → 보안/용량
- 나중에 분석 로직(파이썬)과 화면(JS)을 독립적으로 수정 가능

## 2. 필수 웹 개념 (이것만 알면 시작 가능)

### 2-1. HTTP 요청/응답
브라우저가 서버에 "이거 줘"라고 보내는 편지가 **요청(request)**, 서버가 답장하는 게 **응답(response)**.

요청은 보통 두 정보로 구성됩니다.
- **method**: 무엇을 할지. `GET`(읽기), `POST`(보내기/생성)이 90%.
- **URL/path**: 어디에 요청할지. 예: `http://localhost:8000/api/binned`

### 2-2. REST API & 엔드포인트(endpoint)
서버가 제공하는 각각의 "기능 주소"를 **엔드포인트**라 합니다. 우리 프로젝트 예시:

| Method | Path | 하는 일 |
|--------|------|---------|
| `GET`  | `/api/columns`     | 선택 가능한 feature / target 목록 반환 |
| `POST` | `/api/binned`      | (x_feature, y_target) → 10구간 binning 결과 |
| `POST` | `/api/timeseries`  | (x_feature, y_target) → wafer별 시계열 데이터 |

### 2-3. JSON — 데이터 교환 형식
파이썬 `dict` / `list`와 거의 1:1로 대응되는 텍스트 형식입니다.

```json
{
  "x_feature": "etch_time",
  "bins": [
    {"bin_center": 12.3, "y_avg": 0.81, "wafer_count": 42},
    {"bin_center": 13.7, "y_avg": 0.79, "wafer_count": 51}
  ]
}
```
파이썬에선 `dict`, JS(Vue)에선 `object`로 받습니다. FastAPI가 파이썬 dict → JSON 변환을 자동으로 해줍니다.

### 2-4. 비동기(async) — "기다리는 동안 멈추지 않기"
네트워크 요청은 시간이 걸립니다(수십~수백 ms). 그동안 화면이 얼어붙으면 안 되겠죠.
그래서 JS에서는 `await fetch(...)`처럼 **결과가 올 때까지 기다리되 화면은 살아있게** 하는 비동기 문법을 씁니다.
FastAPI도 `async def`를 지원합니다. 지금은 "요청은 즉시 답이 안 온다 → await로 기다린다" 정도만 알면 충분합니다.

### 2-5. CORS — 처음 만나는 에러 1순위
프론트(`localhost:5173`)와 백엔드(`localhost:8000`)는 **포트가 다르므로 브라우저가 "다른 출처(origin)"로 취급**합니다.
보안상 브라우저는 다른 출처로의 요청을 기본 차단합니다 → 이를 풀어주는 설정이 **CORS**.
FastAPI에서 `CORSMiddleware`로 "5173은 허용"이라고 명시하면 해결됩니다 (6장에서 코드로).

## 3. FastAPI 기초 — 파이썬으로 API 서버 만들기

FastAPI는 파이썬 함수에 데코레이터만 붙이면 API 엔드포인트가 되는 프레임워크입니다.
분석가 친화적인 이유: **타입 힌트(type hint)** 를 그대로 쓰면 입력 검증/문서화가 자동입니다.

아래는 가장 작은 예시입니다. (이 셀은 설명용 — 실제 실행은 `backend/main.py`에서)

In [ ]:
# === 가장 작은 FastAPI 예시 (개념 이해용) ===
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()  # 앱(서버) 객체

# 1) GET 엔드포인트: 그냥 데이터를 돌려줌
@app.get("/api/columns")
def get_columns():
    # 파이썬 dict를 return하면 FastAPI가 자동으로 JSON으로 변환
    return {"features": ["etch_time", "temp", "pressure"], "targets": ["yield"]}

# 2) POST 엔드포인트: 요청 본문(body)을 받음
#    pydantic BaseModel = "받을 데이터의 모양"을 클래스로 선언 → 자동 검증
class BinnedRequest(BaseModel):
    x_feature: str
    y_target: str
    bins: int = 10           # 기본값 10

@app.post("/api/binned")
def make_binned(req: BinnedRequest):
    # req.x_feature, req.y_target 으로 접근 (이미 검증/파싱 완료)
    return {"x_feature": req.x_feature, "received_bins": req.bins}

# 실행은 터미널에서:  uvicorn main:app --reload --port 8000
# 그러면 http://localhost:8000/docs 에서 자동 생성된 API 문서를 볼 수 있음 (매우 유용!)
print("이 셀은 개념 설명용입니다. 실제 서버는 backend/main.py 참고")

> 🎁 **FastAPI의 선물**: 서버를 띄우고 `http://localhost:8000/docs` 에 접속하면
> 모든 엔드포인트를 **버튼으로 직접 테스트**할 수 있는 문서(Swagger UI)가 자동 생성됩니다.
> 프론트엔드를 만들기 전에 API가 잘 동작하는지 여기서 먼저 확인하세요.

## 4. 핵심 분석 로직 — pandas로 binning 계산 (⭐ 실제 실행 셀)

대시보드의 심장은 화려한 차트가 아니라 **올바른 binning 계산**입니다.
이 부분은 익숙한 pandas이므로, 여기서 직접 실행해 결과 dict 모양을 눈으로 확인합시다.
**이 dict가 그대로 JSON이 되어 프론트엔드로 갑니다.**

In [ ]:
import numpy as np
import pandas as pd

# --- 1) 예제 wafer 데이터 만들기 (실제로는 DB/파일에서 읽음) ---
rng = np.random.default_rng(42)
n = 500
df = pd.DataFrame({
    "wafer_id": [f"W{i:04d}" for i in range(n)],
    "etch_time": rng.normal(13, 1.5, n),      # feature 후보 (X)
    "temp":      rng.normal(250, 8, n),       # feature 후보 (X)
    "yield":     rng.normal(0.8, 0.05, n),    # target (Y)
})
# feature와 target에 약한 상관을 부여 (process window가 보이도록)
df["yield"] = (df["yield"] - 0.01 * (df["etch_time"] - 13) ** 2).clip(0, 1)

print(df.head())
print("\n데이터 shape:", df.shape)

In [ ]:
def compute_binned(df: pd.DataFrame, x_feature: str, y_target: str, bins: int = 10) -> dict:
    """X feature를 bins개 등간격 구간으로 나눠 각 구간의 y평균 / wafer수를 계산.

    이 함수의 반환 dict가 그대로 /api/binned 의 JSON 응답이 됩니다.
    """
    x = df[x_feature]
    # pd.cut: 값의 범위를 bins개 '등간격' 구간으로 분할 → 각 행이 어느 구간인지 라벨링
    cats = pd.cut(x, bins=bins)

    grouped = df.groupby(cats, observed=True)[y_target].agg(["mean", "count"])

    result_bins = []
    for interval, row in grouped.iterrows():
        result_bins.append({
            "bin_left":   round(float(interval.left), 4),    # 구간 시작
            "bin_right":  round(float(interval.right), 4),   # 구간 끝
            "bin_center": round(float(interval.mid), 4),     # 구간 중앙 (차트 X좌표로 사용)
            "y_avg":      None if pd.isna(row["mean"]) else round(float(row["mean"]), 4),
            "wafer_count": int(row["count"]),
        })

    return {
        "x_feature": x_feature,
        "y_target": y_target,
        "bins": result_bins,
    }


# --- 실행해서 결과 dict 확인 ---
import json
out = compute_binned(df, x_feature="etch_time", y_target="yield", bins=10)
print(json.dumps(out, indent=2, ensure_ascii=False))

위 출력이 바로 프론트엔드가 받게 될 JSON입니다. 차트는 이 배열을 순회하며
`bin_center`를 X로, `wafer_count`를 막대 높이로, `y_avg`를 선의 높이로 그립니다.

### 시계열 데이터도 같은 원리
시계열 차트는 binning 없이 wafer별 원본 점을 그대로 보냅니다.

In [ ]:
def compute_timeseries(df: pd.DataFrame, x_feature: str, y_target: str) -> dict:
    """wafer별 (시각, target값, feature값)을 시간순으로 반환."""
    # 실제로는 fab_step의 trackout_time 컬럼을 사용. 여기선 예시로 생성.
    times = pd.date_range("2026-01-01", periods=len(df), freq="h")
    d = df.assign(trackout_time=times).sort_values("trackout_time")
    points = [
        {
            "trackout_time": t.isoformat(),
            "y_value": round(float(yv), 4),
            "feature_value": round(float(fv), 4),
        }
        for t, yv, fv in zip(d["trackout_time"], d[y_target], d[x_feature])
    ]
    return {"x_feature": x_feature, "y_target": y_target, "points": points}


ts = compute_timeseries(df, "etch_time", "yield")
print("시계열 점 개수:", len(ts["points"]))
print("첫 점:", ts["points"][0])

## 5. Vue3 기초 — 화면을 그리는 쪽

Vue3는 "데이터가 바뀌면 화면이 자동으로 다시 그려지는(반응형)" 프론트엔드 프레임워크입니다.
우리는 최신 권장 방식인 **Composition API + `<script setup>`** 만 사용합니다.

### 5-1. 컴포넌트 = 화면 조각
`.vue` 파일 하나가 **컴포넌트** 하나입니다. 세 부분으로 구성됩니다.

```vue
<script setup>
  // (1) 로직: 자바스크립트 — 데이터, 함수
  import { ref } from 'vue'
  const count = ref(0)            // ref() = 반응형 변수 (값이 바뀌면 화면 갱신)
  function add() { count.value++ } // ref 값은 .value 로 접근
</script>

<template>
  <!-- (2) 화면: HTML — {{ }} 로 변수 표시, @click 으로 이벤트 -->
  <button @click="add">클릭 수: {{ count }}</button>
</template>

<style scoped>
  /* (3) 디자인: CSS — scoped면 이 컴포넌트에만 적용 */
  button { padding: 8px; }
</style>
```

### 5-2. 꼭 아는 3가지 반응형 도구

| 도구 | 용도 | 예시 |
|------|------|------|
| `ref(x)` | 단일 반응형 값 | `const xFeature = ref('etch_time')` |
| `reactive({})` | 객체 형태 반응형 묶음 | `const spec = reactive({ lower: 10, upper: 16 })` |
| `computed(fn)` | 다른 값에서 파생되는 값 | `const isValid = computed(() => spec.lower < spec.upper)` |

### 5-3. 데이터 흐름 (단방향)
- 부모 → 자식: **props** 로 데이터를 내려줌 (예: App → BinnedChart 에 binning 결과 전달)
- 자식 → 부모: **emit** 으로 이벤트를 올려줌 (예: ControlPanel → App 에 "그리기 눌렀어요")

이 단방향 규칙 덕분에 데이터가 어디서 바뀌는지 추적하기 쉽습니다.

### 5-4. 사용자 입력 양방향 바인딩 — `v-model`
드롭다운/입력창은 `v-model`로 변수와 화면을 자동 동기화합니다. 우리의 ControlPanel 핵심입니다.

```vue
<script setup>
import { ref, reactive } from 'vue'
const xFeature = ref('etch_time')
const yTarget  = ref('yield')
const spec = reactive({ lower: 10, upper: 16 })   // user spec 범위
</script>

<template>
  <!-- 드롭다운 선택값이 xFeature 와 자동 동기화 -->
  <select v-model="xFeature">
    <option>etch_time</option>
    <option>temp</option>
  </select>

  <!-- spec lower ~ upper range filter -->
  <input type="number" v-model.number="spec.lower" />
  <input type="number" v-model.number="spec.upper" />
</template>
```

## 6. 프론트 ↔ 백 연계 — 실제로 데이터 주고받기

### 6-1. axios 로 API 호출
브라우저 내장 `fetch`도 되지만, 코드가 깔끔한 `axios`를 권장합니다.

```js
// frontend/src/api/client.js  — API 호출을 한 곳에 모음
import axios from 'axios'

const api = axios.create({ baseURL: '/api' })   // 모든 요청 앞에 /api 를 붙임

export async function fetchColumns() {
  const res = await api.get('/columns')          // GET /api/columns
  return res.data                                // 서버가 준 JSON (dict)
}

export async function fetchBinned(xFeature, yTarget, bins = 10) {
  const res = await api.post('/binned', {        // POST /api/binned + body
    x_feature: xFeature, y_target: yTarget, bins,
  })
  return res.data
}
```

> 핵심: `res.data`가 4장에서 만든 그 dict(JSON)입니다. 파이썬에서 보낸 게 그대로 도착합니다.

### 6-2. Vite proxy — CORS를 우아하게 피하기
개발 중엔 프론트(5173)에서 `/api/...` 로 요청하면 Vite가 백엔드(8000)로 **몰래 전달**해 줍니다.
브라우저 입장에선 같은 출처(5173)로 보이므로 CORS 문제가 사라집니다.

```js
// frontend/vite.config.js
export default defineConfig({
  plugins: [vue()],
  server: {
    proxy: {
      '/api': 'http://localhost:8000',   // /api 로 시작하는 요청 → 8000번으로 전달
    },
  },
})
```

### 6-3. 그래도 백엔드엔 CORS 설정을 켜둔다
배포 환경 등 proxy를 못 쓰는 경우를 대비해 FastAPI에도 허용 설정을 둡니다.

```python
# backend/main.py
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],   # 프론트 개발 서버 허용
    allow_methods=["*"], allow_headers=["*"],
)
```

### 6-4. 전체 데이터 흐름 한눈에 보기
```
[사용자가 feature/target 선택 + '차트 그리기' 클릭]
        │  (Vue: @click 이벤트)
        ▼
fetchBinned('etch_time','yield')      ← axios POST /api/binned
        │  (Vite proxy 통해 8000으로)
        ▼
FastAPI make_binned()  →  compute_binned(df,...)   ← 4장의 그 함수!
        │  (dict 반환 → 자동 JSON 변환)
        ▼
res.data = { bins: [...] }            ← Vue가 받음
        │  (ref 변수에 저장 → 반응형 트리거)
        ▼
BinnedChart 컴포넌트가 ECharts로 다시 그림
```

## 7. 차트 — ECharts로 Combination Chart 그리기

차트 라이브러리는 여러 개(Chart.js, Plotly, ECharts)가 있지만,
**막대+선 혼합, 수직선(markLine), 차트 간 연동**을 모두 깔끔히 지원하는 **ECharts**를 추천합니다.

설치: `npm install echarts vue-echarts`

### 7-1. ECharts의 핵심 개념: `option` 객체 하나로 모든 걸 기술
ECharts는 "어떻게 그릴지"를 거대한 **설정 객체(option)** 로 표현합니다.
데이터가 바뀌면 option만 새로 만들어 넣으면 차트가 갱신됩니다.

### 7-2. Combination chart (bar = wafer count, line = y avg) + spec 수직선
아래는 BinnedChart 컴포넌트가 만드는 option의 핵심입니다.

In [ ]:
# (참고용 — 이건 JavaScript 코드입니다. 노트북에선 실행하지 않습니다.)
js_binned_option = r"""
function buildBinnedOption(binned, spec) {
  const centers = binned.bins.map(b => b.bin_center)
  const counts  = binned.bins.map(b => b.wafer_count)
  const yavgs   = binned.bins.map(b => b.y_avg)

  return {
    tooltip: { trigger: 'axis' },          // 마우스 올리면 값 표시 (인터랙티브)
    legend:  { data: ['wafer count', 'y avg'] },
    xAxis: {
      type: 'value',
      name: binned.x_feature,
      // bin_center 들을 X좌표로 사용 → 진짜 feature value 축
    },
    yAxis: [
      { type: 'value', name: 'wafer count' },                 // 왼쪽 축 (막대)
      { type: 'value', name: 'y avg', position: 'right' },    // 오른쪽 축 (선)
    ],
    series: [
      {
        name: 'wafer count', type: 'bar', yAxisIndex: 0,
        data: centers.map((c, i) => [c, counts[i]]),
      },
      {
        name: 'y avg', type: 'line', yAxisIndex: 1, smooth: true,
        data: centers.map((c, i) => [c, yavgs[i]]),
        // ⭐ spec lower/upper 를 수직선(markLine)으로 표시
        markLine: {
          symbol: 'none',
          data: [
            { xAxis: spec.lower, label: { formatter: 'spec L' }, lineStyle: { color: 'red' } },
            { xAxis: spec.upper, label: { formatter: 'spec U' }, lineStyle: { color: 'red' } },
          ],
        },
      },
    ],
  }
}
"""
print("BinnedChart.vue 의 핵심 option 빌더 (frontend 폴더 참고)")

포인트 정리:
- **`type: 'bar'` + `type: 'line'`** 을 한 차트의 `series`에 같이 넣으면 combination chart가 됩니다.
- **이중 Y축**: `yAxis`를 배열로 주고 series에서 `yAxisIndex`로 연결 → 막대(개수)와 선(평균)의 스케일 분리.
- **spec 수직선**: line series의 `markLine`에 `{ xAxis: 값 }` 을 주면 그 X위치에 세로선이 그어집니다.
  spec range filter(lower/upper)가 바뀔 때마다 option을 다시 만들면 수직선이 즉시 이동합니다 → **인터랙티브**.
- **인터랙티브**: `tooltip`, 그리고 `dataZoom`(확대/스크롤)을 추가하면 사용자가 직접 탐색 가능.

## 8. 시계열 Dual Scatter — 두 차트를 시간축으로 연동

요구사항: X축 = `trackout_time`, 위 차트 Y = target(y), 아래 차트 Y = feature.
그리고 두 차트가 **같은 시간축으로 함께 움직여야** 합니다(확대/마우스 동기화).

ECharts에서는 **하나의 차트에 grid(격자) 2개를 위아래로 배치**하고, 같은 시간축을 공유하면 깔끔합니다.

In [ ]:
js_timeseries_option = r"""
function buildTimeseriesOption(ts) {
  const yPoints = ts.points.map(p => [p.trackout_time, p.y_value])
  const fPoints = ts.points.map(p => [p.trackout_time, p.feature_value])

  return {
    // axisPointer.link: 두 grid의 십자선(crosshair)을 시간축으로 동기화
    axisPointer: { link: [{ xAxisIndex: 'all' }] },
    tooltip: { trigger: 'axis' },
    // grid 2개: 위(top) / 아래(bottom)
    grid: [
      { left: 60, right: 30, top: 40,  height: '32%' },   // 위 차트 영역
      { left: 60, right: 30, top: '58%', height: '32%' }, // 아래 차트 영역
    ],
    xAxis: [
      { type: 'time', gridIndex: 0 },                       // 위 차트 X (시간)
      { type: 'time', gridIndex: 1 },                       // 아래 차트 X (시간)
    ],
    yAxis: [
      { type: 'value', gridIndex: 0, name: ts.y_target },   // 위: y value
      { type: 'value', gridIndex: 1, name: ts.x_feature },  // 아래: feature value
    ],
    // 두 차트를 같이 확대/스크롤
    dataZoom: [{ type: 'inside', xAxisIndex: [0, 1] }, { type: 'slider', xAxisIndex: [0, 1] }],
    series: [
      { name: 'y',       type: 'scatter', xAxisIndex: 0, yAxisIndex: 0, data: yPoints, symbolSize: 5 },
      { name: 'feature', type: 'scatter', xAxisIndex: 1, yAxisIndex: 1, data: fPoints, symbolSize: 5 },
    ],
  }
}
"""
print("TimeSeriesChart.vue 의 핵심 option 빌더")

포인트 정리:
- **grid 2개**로 위/아래 영역을 나눕니다. 각각 자신의 `xAxisIndex`, `yAxisIndex`를 가집니다.
- **`axisPointer.link`** 로 두 차트의 십자선이 같은 시간에 묶입니다 → 위에서 본 wafer를 아래에서도 즉시 확인.
- **`dataZoom`에 `xAxisIndex: [0, 1]`** → 한쪽을 확대하면 둘 다 같은 시간 구간으로 확대됩니다.
- `type: 'time'` 축은 ISO 문자열(`2026-01-01T00:00:00`)을 자동으로 시간으로 해석합니다 (4장에서 `isoformat()`으로 보낸 이유).

## 9. 파일 아키텍처 — 프로젝트 폴더 구조

옆 폴더에 아래 구조로 **실제 동작하는 스캐폴드**를 함께 만들어 두었습니다.

```
prc_window/
├── process_window_dashboard_guide.ipynb   ← (이 노트북)
│
├── backend/                         # FastAPI (파이썬) — 데이터 계산
│   ├── main.py                      # 앱 생성 + CORS + 엔드포인트 라우팅
│   ├── analytics.py                 # ⭐ compute_binned / compute_timeseries (4장 로직)
│   ├── data.py                      # 예제 wafer 데이터 로딩 (→ 실제 DB/파일로 교체)
│   ├── schemas.py                   # pydantic 요청/응답 모델
│   └── requirements.txt             # fastapi, uvicorn, pandas ...
│
└── frontend/                        # Vue3 + Vite — 화면/차트
    ├── index.html                   # 진입 HTML
    ├── package.json                 # 의존성 (vue, echarts, axios)
    ├── vite.config.js               # 빌드 + /api proxy 설정
    └── src/
        ├── main.js                  # Vue 앱 부트스트랩
        ├── App.vue                  # 전체 레이아웃 + 상태(state) 보유 (부모)
        ├── api/
        │   └── client.js            # axios 호출 모음 (fetchColumns/Binned/Timeseries)
        └── components/
            ├── ControlPanel.vue     # feature/target 선택 + spec range filter
            ├── BinnedChart.vue      # 7장: combination chart + 수직선
            └── TimeSeriesChart.vue  # 8장: dual scatter
```

### 폴더 구조의 원칙 (왜 이렇게 나누나)
- **backend / frontend 완전 분리**: 서로 다른 언어/실행 환경. 각자 독립 실행·배포.
- **백엔드 내부 분리**:
  - `main.py`(라우팅, 입출력) ↔ `analytics.py`(순수 계산) 분리 → 계산 로직을 노트북에서 테스트한 그대로 재사용.
  - `schemas.py`로 "주고받는 데이터 모양"을 한 곳에 모음 → 프론트와의 계약서 역할.
- **프론트 내부 분리**:
  - `api/client.js`: 네트워크 호출을 **한 파일에 격리**. 차트 컴포넌트는 "데이터가 어디서 오는지" 신경 안 씀.
  - `components/`: 화면 조각 단위로 파일 분리. 각 컴포넌트는 props를 받아 그리기만 함.
  - `App.vue`가 **상태(현재 선택, spec, 받은 데이터)를 보유**하고 자식에게 내려줌(단방향 흐름, 5-3 참고).

## 10. 실행 방법 — 실제로 띄워 보기

터미널 **2개**를 엽니다 (백엔드용 1개, 프론트엔드용 1개).

### ① 백엔드 (FastAPI)
```bash
cd backend
python3 -m venv .venv && source .venv/bin/activate   # 가상환경 (선택이지만 권장)
pip install -r requirements.txt
uvicorn main:app --reload --port 8000
# → http://localhost:8000/docs 에서 API 먼저 테스트해 보세요
```

### ② 프론트엔드 (Vue3 + Vite)
```bash
cd frontend
npm install
npm run dev
# → http://localhost:5173 접속
```

### 동작 확인 체크리스트
1. `http://localhost:8000/docs` 에서 `/api/binned`를 직접 실행 → JSON이 나오는가? (백엔드 OK)
2. `http://localhost:5173` 접속 → feature/target 드롭다운이 채워지는가? (연계 OK)
3. feature/target 선택 후 "차트 그리기" → combination chart가 뜨는가?
4. spec lower/upper 입력 → 빨간 수직선이 움직이는가?
5. 시계열 차트에서 확대(드래그/스크롤) → 위·아래가 같이 움직이는가?

### 다음 단계 (학습 확장)
- `data.py`의 예제 데이터를 **실제 wafer 데이터**(CSV/DB)로 교체
- feature를 여러 개 선택하는 multi-select, 차트 다운로드 버튼 추가
- 로딩 스피너 / 에러 메시지 표시 (UX)
- 배포: 프론트는 `npm run build` 후 정적 파일, 백엔드는 uvicorn/gunicorn

---
수고했습니다 🎉 이 노트북의 4장(분석 로직)을 손에 익히는 게 가장 중요합니다.
나머지(FastAPI 껍데기 + Vue 차트)는 옆 폴더 코드를 보고 따라 고치면서 익히세요.